# Luminex SPSA Tuning Job
Self-contained: clones repo, builds engine, installs cutechess, runs SPSA tuning.
Run this as a training job (non-interactive).

In [ ]:
import subprocess
import os
import sys
import time
import json

def run(cmd, cwd=None, check=True):
    print(f">>> {cmd}")
    result = subprocess.run(cmd, shell=True, cwd=cwd, check=False,
                           capture_output=False, text=True)
    if check and result.returncode != 0:
        print(f"FAILED (exit {result.returncode}): {cmd}")
        sys.exit(1)
    return result.returncode

WORK_DIR = os.path.expanduser("~/Luminex")
print(f"Work directory: {WORK_DIR}")

In [ ]:
# Step 1: Install build dependencies
print("=== Installing build dependencies ===")
run("sudo apt-get update -qq 2>/dev/null || true")
run("sudo apt-get install -y -qq cmake g++ make git 2>/dev/null || true")
run("sudo apt-get install -y -qq cutechess-cli 2>/dev/null || true")
print("Dependencies installed.")

In [ ]:
# Step 2: Clone and build engine
print("=== Building Luminex ===")
if not os.path.exists(WORK_DIR):
    run(f"git clone https://github.com/changcheng967/Luminex.git {WORK_DIR}")
else:
    run(f"cd {WORK_DIR} && git pull || true")

build_dir = os.path.join(WORK_DIR, "build")
os.makedirs(build_dir, exist_ok=True)
run(f"cd {build_dir} && cmake .. -DCMAKE_BUILD_TYPE=Release", cwd=WORK_DIR)
run(f"cmake --build {build_dir} -j$(nproc)", cwd=WORK_DIR)

engine = os.path.join(build_dir, "luminex")
assert os.path.exists(engine), f"Engine not found at {engine}"
print(f"Engine built: {engine}")

# Quick sanity check
run(f"echo 'uci' | {engine} | head -5", check=False)

In [ ]:
# Step 3: Find cutechess-cli
print("=== Finding cutechess-cli ===")
result = subprocess.run("which cutechess-cli 2>/dev/null || find / -name 'cutechess-cli' -type f 2>/dev/null | head -1",
                       shell=True, capture_output=True, text=True, timeout=30)
cutechess = result.stdout.strip()

if not cutechess:
    print("cutechess-cli not found, downloading...")
    import urllib.request
    install_dir = os.path.expanduser("~/cutechess")
    os.makedirs(install_dir, exist_ok=True)
    tar_path = os.path.join(install_dir, "cutechess.tar.gz")
    urllib.request.urlretrieve(
        "https://github.com/cutechess/cutechess/releases/download/cli-1.3.1/cutechess-cli-1.3.1-linux.tar.gz",
        tar_path)
    run(f"tar xzf {tar_path} -C {install_dir}")
    result2 = subprocess.run(f"find {install_dir} -name 'cutechess-cli' -type f",
                            shell=True, capture_output=True, text=True)
    cutechess = result2.stdout.strip().split('\n')[0] if result2.stdout.strip() else ""
    if cutechess:
        run(f"chmod +x {cutechess}")

assert cutechess, "Could not find or install cutechess-cli!"
print(f"cutechess-cli: {cutechess}")

In [ ]:
# Step 4: Run SPSA tuning (50K iterations)
print("=== Starting SPSA Tuning ===")

iterations = 50000
output = os.path.join(WORK_DIR, "spsa_result.json")

cmd = (
    f"python3 {WORK_DIR}/tools/spsa_tune.py "
    f"--engine {engine} "
    f"--cutechess {cutechess} "
    f"--iterations {iterations} "
    f"--rounds 2 "
    f"--tc 1+0.01 "
    f"--seed 42 "
    f"--output {output}"
)

# Resume from previous run if exists
if os.path.exists(output):
    cmd += f" --resume {output}"
    print(f"Resuming from {output}")

print(f"Command: {cmd}")
print(f"This will take several hours. Iterations: {iterations}")
print("=" * 60)

start = time.time()
run(cmd, cwd=WORK_DIR)
elapsed = time.time() - start
print(f"\nSPSA completed in {elapsed/3600:.1f} hours")

In [ ]:
# Step 5: Display results
print("=" * 60)
print("SPSA TUNING RESULTS")
print("=" * 60)

if os.path.exists(output):
    with open(output) as f:
        data = json.load(f)
    print(f"Iterations completed: {data['iteration']}")
    print(f"\n# Final parameters (UCI setoption format):")
    for p in data['params']:
        old = p.get('initial', p['theta'])
        new = int(round(p['theta']))
        chg = new - int(round(old))
        print(f"setoption name {p['name']} value {new}  # chg={chg:+d}")
    print(f"\nResults file: {output}")
else:
    print(f"No results file found at {output}")